In [1]:
# Force TF backend, fix seeds so the sweep is reproducible, and stay on CPU.
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import tensorflow as tf
from ee446_data import (
    load_export, make_windows, group_train_test_split, summarize,
    TARGET_LABELS, WINDOW_SAMPLES,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.set_visible_devices([], "GPU")   # CPU only: keeps results deterministic

print("TF:", tf.__version__, "| NumPy:", np.__version__)

TF: 2.14.1 | NumPy: 1.26.4


In [2]:
# Read the raw CBOR recordings exported from Edge Impulse Studio.
recs = load_export("ee-446-final-project-export")

Found 1033 sample files under ee-446-final-project-export  (1033 .cbor, 0 .json)

Loaded 1033 samples across 13 labels:
  Circle         61
  Down           60
  Down1         120
  Idle           91
  Left           64
  Left1         120
  Right          61
  Right1        125
  Square         76
  Triangle       60
  Up             60
  Up1           120
  testing        15


In [3]:
# Slice recordings into 1500 ms / 100 ms-stride windows, keep only the 5 active classes,
# then split GROUP-AWARE: all windows from one recording stay on the same side,
# so overlapping windows can never leak across the train/test boundary.
Xt, yt, gt, target_classes = make_windows(recs, TARGET_LABELS)
Xt_tr, yt_tr, Xt_te, yt_te = group_train_test_split(Xt, yt, gt, test_size=0.2, seed=SEED)
summarize(yt_tr, target_classes, "target train")
summarize(yt_te, target_classes, "target test")

Windows: (3456, 150, 6)  (window=150, stride=10, scale=0.05)
  Down1          720
  Idle           546
  Left1          720
  Right1         750
  Up1            720
Recordings: 576  ->  train 461 / test 115
Windows:    train 2766 / test 690
target train: 2766 windows
  Down1          582  (21.0%)
  Idle           432  (15.6%)
  Left1          534  (19.3%)
  Right1         606  (21.9%)
  Up1            612  (22.1%)
target test: 690 windows
  Down1          138  (20.0%)
  Idle           114  (16.5%)
  Left1          186  (27.0%)
  Right1         144  (20.9%)
  Up1            108  (15.7%)


In [4]:
# Edge Impulse Python SDK: used later to estimate RAM/flash/latency on the Nano.
import sys
!{sys.executable} -m pip install edgeimpulse

  Using cached edgeimpulse-1.0.18-py3-none-any.whl.metadata (2.6 kB)
  Using cached edgeimpulse_api-1.86.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached python_socketio-5.16.4-py3-none-any.whl.metadata (3.3 kB)
  Using cached aenum-3.1.17-py3-none-any.whl.metadata (3.8 kB)
  Using cached urllib3-1.26.20-py2.py3-none-any.whl.metadata (50 kB)
  Using cached bidict-0.23.1-py3-none-any.whl.metadata (8.7 kB)
  Using cached python_engineio-4.13.5-py3-none-any.whl.metadata (1.9 kB)
  Using cached simple_websocket-1.1.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
Using cached edgeimpulse-1.0.18-py3-none-any.whl (68 kB)
Using cached edgeimpulse_api-1.86.0-py3-none-any.whl (1.6 MB)
Using cached aenum-3.1.17-py3-none-any.whl (165 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.2 MB/s  0:00:00
Using cached python_socketio-5.16.4-py3-none-any.whl (82 kB)
Using cached urllib3-1.26.20-py2.py3-none-any.whl (144 kB)
Using cached b

In [3]:
#The profiling cells require an Edge Impulse API key. either use this or Create one in Studio
# (Dashboard → Keys), then run in a terminal: echo "your-key" > ~/.ei_api_key All other cells run without it.
!echo "ei_eebe54886f274dbb17ac6ec990b92f88ed4e5c37117f562a" > ~/.ei_api_key

In [8]:
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers.legacy import Adam

INPUT_SHAPE = (WINDOW_SAMPLES, 6)   # (150 samples, 6 IMU axes)

def build_cnn(num_classes, width=1.0, name="cnn"):
    """Conv1D stack with a width multiplier. width=1.0 -> 16/32/64 (Impulse #2 clone).
    Only the filter counts scale; kernels, pooling and the dense head stay fixed,
    so the sweep isolates width as the single variable."""
    filters = [max(2, int(round(f * width))) for f in (16, 32, 64)]
    kernels = [5, 5, 3]
    inp = Input(shape=INPUT_SHAPE, name="inp")
    x = inp
    for i, (f, k) in enumerate(zip(filters, kernels), start=1):
        x = layers.Conv1D(f, k, padding="same", activation="relu", name=f"conv{i}")(x)
        x = layers.BatchNormalization(name=f"bn{i}")(x)
        x = layers.MaxPooling1D(2, name=f"pool{i}")(x)
        x = layers.Dropout(0.25, name=f"drop{i}")(x)
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(32, activation="relu", name="dense1")(x)
    x = layers.Dropout(0.4, name="dropout1")(x)
    out = layers.Dense(num_classes, activation="softmax", name="classifier")(x)
    return Model(inp, out, name=name)

def compile_model(m, lr=5e-4):
    # Same optimizer/loss as the Edge Impulse impulse, for a fair comparison.
    m.compile(optimizer=Adam(learning_rate=lr),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

# Stop early once validation loss stops improving; restore the best weights.
early = lambda: tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True)

In [9]:
# Train three widths of the same architecture: full, half, quarter.
# Re-seeding before each run keeps the comparison fair.
os.makedirs("models", exist_ok=True)

WIDTHS = {"full_16_32_64": 1.0, "half_8_16_32": 0.5, "quarter_4_8_16": 0.25}
models_fp32, acc_fp32 = {}, {}

for name, w in WIDTHS.items():
    tf.random.set_seed(SEED)
    m = compile_model(build_cnn(len(target_classes), width=w, name=name))
    m.fit(Xt_tr, yt_tr, epochs=60, batch_size=32,
          validation_split=0.15, callbacks=[early()], verbose=0)
    loss, acc = m.evaluate(Xt_te, yt_te, verbose=0)
    models_fp32[name], acc_fp32[name] = m, acc
    m.save(f"models/{name}.h5")
    print(f"{name:18s} params = {m.count_params():>7,d}   test acc = {acc:.4f}")

/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


full_16_32_64      params =  46,805   test acc = 0.9884
half_8_16_32       params =  21,325   test acc = 0.9899
quarter_4_8_16     params =  10,217   test acc = 0.9928


In [10]:
def to_tflite(model, int8=False):
    """Convert to TFLite. With int8=True this is post-training quantization (PTQ):
    a representative sample of training windows calibrates the activation ranges."""
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if int8:
        def rep_data():
            for i in range(0, len(Xt_tr), 25):
                yield [Xt_tr[i:i+1].astype(np.float32)]
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = rep_data
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8
        conv.inference_output_type = tf.int8
    return conv.convert()

def eval_tflite(tfl_bytes, X, y):
    """Run the converted model window by window; quantize inputs the same way
    the device would, so this measures true deployed-format accuracy."""
    interp = tf.lite.Interpreter(model_content=tfl_bytes)
    interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    correct = 0
    for i in range(len(X)):
        x = X[i:i+1].astype(np.float32)
        if inp["dtype"] == np.int8:
            s, z = inp["quantization"]
            x = (x / s + z).round().clip(-128, 127).astype(np.int8)
        interp.set_tensor(inp["index"], x)
        interp.invoke()
        correct += int(interp.get_tensor(out["index"])[0].argmax() == y[i])
    return correct / len(X)

# Quantize every sweep model and compare fp32 vs int8 accuracy and file size.
tflite_models = {}
print(f"{'config':18s} {'fp32 acc':>9s} {'int8 acc':>9s} {'fp32 KB':>8s} {'int8 KB':>8s}")
for name, m in models_fp32.items():
    fp32 = to_tflite(m, int8=False)
    int8 = to_tflite(m, int8=True)
    tflite_models[name] = int8
    with open(f"models/{name}_int8.tflite", "wb") as f:
        f.write(int8)
    a8 = eval_tflite(int8, Xt_te, yt_te)
    print(f"{name:18s} {acc_fp32[name]:>9.4f} {a8:>9.4f} {len(fp32)/1024:>8.1f} {len(int8)/1024:>8.1f}")

config              fp32 acc  int8 acc  fp32 KB  int8 KB
INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpgfew2dxj/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpgfew2dxj/assets
2026-08-18 18:59:06.033437: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-18 18:59:06.033460: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-18 18:59:06.033820: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpgfew2dxj
2026-08-18 18:59:06.035058: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-18 18:59:06.035062: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpgfew2dxj
2026-08-18 18:59:06.038339: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-08-18 18:59:06.039451: I tensorflow/cc/saved_model/load

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmp1t1_ci3j/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmp1t1_ci3j/assets
/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-08-18 18:59:06.955520: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-18 18:59:06.955533: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-18 18:59:06.955625: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmp1t1_ci3j
2026-08-18 18:59:06.956934: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-18 18:59:06.956939: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77

full_16_32_64         0.9884    0.9899    192.0     60.0
INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpw6pet6p1/assets


...

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmppe6886hv/assets
2026-08-18 18:59:09.676885: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-18 18:59:09.676897: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-18 18:59:09.676992: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmppe6886hv
2026-08-18 18:59:09.678238: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-18 18:59:09.678242: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmppe6886hv
2026-08-18 18:59:09.681707: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-08-18 18:59:09.716812: I tensorflow/cc/saved_model/loader.cc:217] Running initialization

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpw5f08hrx/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpw5f08hrx/assets


quarter_4_8_16        0.9928    0.9928     49.9     22.4


/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-08-18 18:59:10.557579: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-18 18:59:10.557596: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-18 18:59:10.557700: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpw5f08hrx
2026-08-18 18:59:10.558847: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-18 18:59:10.558853: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpw5f08hrx
2026-08-18 18:59:10.561966: I tensorflow/cc/saved_model/loader.cc:233] 

In [13]:
# Magnitude pruning: gradually zero the smallest 50% of weights during a short
# fine-tune, so the network can recover accuracy as sparsity increases.
import tensorflow_model_optimization as tfmot

base = models_fp32["half_8_16_32"]
end_step = int(np.ceil(len(Xt_tr) * 0.85 / 32)) * 20   # steps in 20 epochs

pruned = tfmot.sparsity.keras.prune_low_magnitude(
    base,
    pruning_schedule=tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0, final_sparsity=0.5,
        begin_step=0, end_step=end_step))

compile_model(pruned, lr=1e-4)   # low LR: recovery, not retraining
pruned.fit(Xt_tr, yt_tr, epochs=20, batch_size=32, validation_split=0.15,
           callbacks=[tfmot.sparsity.keras.UpdatePruningStep()], verbose=0)

# strip_pruning removes the pruning wrappers, leaving a plain sparse model.
stripped = tfmot.sparsity.keras.strip_pruning(pruned)
loss, acc_pruned = stripped.evaluate(Xt_te, yt_te, verbose=0)
stripped.save("models/half_pruned50.h5")

int8_pruned = to_tflite(stripped, int8=True)
with open("models/half_pruned50_int8.tflite", "wb") as f:
    f.write(int8_pruned)
a8p = eval_tflite(int8_pruned, Xt_te, yt_te)
print(f"half + 50% prune   fp32 acc = {acc_pruned:.4f}   int8 acc = {a8p:.4f}   int8 KB = {len(int8_pruned)/1024:.1f}")

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.

In [14]:
# strip_pruning returns an uncompiled model, so the evaluation above fails on a
# fresh kernel; recompile and repeat the evaluation, export and INT8 conversion here.
compile_model(stripped)
loss, acc_pruned = stripped.evaluate(Xt_te, yt_te, verbose=0)
stripped.save("models/half_pruned50.h5")

int8_pruned = to_tflite(stripped, int8=True)
with open("models/half_pruned50_int8.tflite", "wb") as f:
    f.write(int8_pruned)
a8p = eval_tflite(int8_pruned, Xt_te, yt_te)
print(f"half + 50% prune   fp32 acc = {acc_pruned:.4f}   int8 acc = {a8p:.4f}   int8 KB = {len(int8_pruned)/1024:.1f}")

/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpylxnggn0/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpylxnggn0/assets
/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


half + 50% prune   fp32 acc = 0.9884   int8 acc = 0.9884   int8 KB = 33.7


2026-08-18 19:05:52.409849: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-18 19:05:52.409865: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-18 19:05:52.409962: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpylxnggn0
2026-08-18 19:05:52.410991: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-18 19:05:52.410996: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpylxnggn0
2026-08-18 19:05:52.414428: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-08-18 19:05:52.433417: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpylxnggn0
2026-08-

In [15]:
# TFLite stores zeroed weights densely, so the raw file does not shrink.
# Comparing gzipped sizes shows the sparsity is real but only pays off
# under compressed storage.
import gzip
for fname in ["half_8_16_32_int8.tflite", "half_pruned50_int8.tflite"]:
    raw = open(f"models/{fname}", "rb").read()
    gz = len(gzip.compress(raw))
    print(f"{fname:32s} raw {len(raw)/1024:6.1f} KB   gzipped {gz/1024:6.1f} KB")

half_8_16_32_int8.tflite         raw   33.7 KB   gzipped   23.3 KB
half_pruned50_int8.tflite        raw   33.7 KB   gzipped   17.7 KB


In [18]:
#The profiling cells require an Edge Impulse API key. either use this or Create one in Studio
# (Dashboard → Keys), then run in a terminal: echo "your-key" > ~/.ei_api_key All other cells run without it.
!echo "ei_063c48a4197defd3570e19736b33aa5cdae879dfe0e60330" > ~/.ei_api_key

In [20]:
# Estimate RAM / flash / latency on the Arduino Nano 33 BLE using the same
# Edge Impulse profiler that produced the Studio figures for both impulses.
# The API key is read from a file outside the repo, never hard-coded.
import edgeimpulse as ei
ei.API_KEY = open(os.path.expanduser("~/.ei_api_key")).read().strip()

profile_results = {}
for fname in ["full_16_32_64_int8", "half_8_16_32_int8",
              "quarter_4_8_16_int8", "half_pruned50_int8"]:
    try:
        res = ei.model.profile(model=f"models/{fname}.tflite",
                               device="arduino-nano-33-ble")
        profile_results[fname] = res
        print(f"=== {fname} ===")
        print(res.summary())
    except Exception as e:
        print(f"{fname}: FAILED -> {e}")

=== full_16_32_64_int8 ===
Target results for int8:
{
    "variant": "int8",
    "device": "arduino-nano-33-ble",
    "tfliteFileSizeBytes": 61440,
    "isSupportedOnMcu": true,
    "memory": {
        "tflite": {
            "ram": 15096,
            "rom": 119800,
            "arenaSize": 14656
        },
        "eon": {
            "ram": 10656,
            "rom": 91168,
            "arenaSize": 7440
        }
    },
    "timePerInferenceMs": 26,
    "customMetrics": [],
    "hasPerformance": true
}


Performance on device types:
{
    "variant": "int8",
    "lowEndMcu": {
        "description": "Estimate for a Cortex-M0+ or similar, running at 40MHz",
        "timePerInferenceMs": 213,
        "memory": {
            "tflite": {
                "ram": 14904,
                "rom": 103752
            },
            "eon": {
                "ram": 9696,
                "rom": 77856
            }
        },
        "supported": true
    },
    "highEndMcu": {
        "description": "

In [19]:
raw = open(os.path.expanduser("~/.ei_api_key")).read()
print(repr(raw[:8]), "len =", len(raw.strip()))

'ei_063c4' len = 51


In [21]:
# Two probes below quarter width, to find where accuracy actually breaks.
extra = {}

# eighth width: 2/4/8 filters -- same depth, very narrow first layer
tf.random.set_seed(SEED)
m8 = compile_model(build_cnn(len(target_classes), width=0.125, name="eighth_2_4_8"))
m8.fit(Xt_tr, yt_tr, epochs=60, batch_size=32, validation_split=0.15,
       callbacks=[early()], verbose=0)
extra["eighth_2_4_8"] = m8

def build_minimal(num_classes):
    """Same parameter budget as 'eighth', but spent differently: one conv layer
    that is 8 filters WIDE instead of three starved layers. Tests whether the
    limit is total capacity or first-layer width."""
    inp = Input(shape=INPUT_SHAPE, name="inp")
    x = layers.Conv1D(8, 5, padding="same", activation="relu", name="conv1")(inp)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling1D(4, name="pool1")(x)
    x = layers.Dropout(0.25, name="drop1")(x)
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(16, activation="relu", name="dense1")(x)
    out = layers.Dense(num_classes, activation="softmax", name="classifier")(x)
    return Model(inp, out, name="minimal_1conv")

tf.random.set_seed(SEED)
mm = compile_model(build_minimal(len(target_classes)))
mm.fit(Xt_tr, yt_tr, epochs=60, batch_size=32, validation_split=0.15,
       callbacks=[early()], verbose=0)
extra["minimal_1conv"] = mm

# Evaluate, save and quantize both probes.
for name, m in extra.items():
    loss, acc = m.evaluate(Xt_te, yt_te, verbose=0)
    m.save(f"models/{name}.h5")
    int8 = to_tflite(m, int8=True)
    with open(f"models/{name}_int8.tflite", "wb") as f:
        f.write(int8)
    a8 = eval_tflite(int8, Xt_te, yt_te)
    models_fp32[name], acc_fp32[name] = m, acc
    print(f"{name:16s} params = {m.count_params():>7,d}   fp32 = {acc:.4f}   int8 = {a8:.4f}   int8 KB = {len(int8)/1024:.1f}")

/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpipprgho_/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpipprgho_/assets
/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-08-18 19:22:00.851274: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-18 19:22:00.851285: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-18 19:22:00.851389: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpipprgho_
2026-08-18 19:22:00.852827: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-18 19:22:00.852831: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77

eighth_2_4_8     params =   5,071   fp32 = 0.8913   int8 = 0.8913   int8 KB = 16.8
INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpcroqhlod/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpcroqhlod/assets
/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-08-18 19:22:01.360316: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-18 19:22:01.360327: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-18 19:22:01.360417: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpcroqhlod
2026-08-18 19:22:01.361080: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-18 19:22:01.361084: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77

minimal_1conv    params =   5,117   fp32 = 0.9928   int8 = 0.9928   int8 KB = 10.6


In [22]:
# Same profiler call for the two extra configurations.
for fname in ["eighth_2_4_8_int8", "minimal_1conv_int8"]:
    res = ei.model.profile(model=f"models/{fname}.tflite",
                           device="arduino-nano-33-ble")
    profile_results[fname] = res
    print(f"=== {fname} ===")
    print(res.summary())

=== eighth_2_4_8_int8 ===
Target results for int8:
{
    "variant": "int8",
    "device": "arduino-nano-33-ble",
    "tfliteFileSizeBytes": 17168,
    "isSupportedOnMcu": true,
    "memory": {
        "tflite": {
            "ram": 10603,
            "rom": 75528,
            "arenaSize": 10163
        },
        "eon": {
            "ram": 6272,
            "rom": 48192,
            "arenaSize": 3056
        }
    },
    "timePerInferenceMs": 23,
    "customMetrics": [],
    "hasPerformance": true
}


Performance on device types:
{
    "variant": "int8",
    "lowEndMcu": {
        "description": "Estimate for a Cortex-M0+ or similar, running at 40MHz",
        "timePerInferenceMs": 180,
        "memory": {
            "tflite": {
                "ram": 10411,
                "rom": 59480
            },
            "eon": {
                "ram": 5920,
                "rom": 34864
            }
        },
        "supported": true
    },
    "highEndMcu": {
        "description": "Esti

In [23]:
# Collected results: accuracy, file size, and Nano figures (EON compiler) for
# every configuration, with the two impulses as reference points.
rows = [
    # name,                acc,    file KB, EON RAM KB, EON ROM KB, latency ms
    ("full_16_32_64",     0.9899,  60.0,  10.4,  89.0, 26),
    ("half_8_16_32",      0.9899,  33.7,   7.3,  63.4, 14),
    ("quarter_4_8_16",    0.9928,  22.4,   6.3,  52.3, 15),
    ("half_pruned50",     0.9884,  33.7,   7.3,  63.4, 17),
    ("eighth_2_4_8",      0.8913,  16.8,   6.1,  47.1, 23),
    ("minimal_1conv",     0.9928,  10.6,   4.8,  45.7,  9),
]
print(f"{'config':16s} {'int8 acc':>8s} {'file KB':>8s} {'RAM KB':>7s} {'ROM KB':>7s} {'lat ms':>7s}")
for r in rows:
    print(f"{r[0]:16s} {r[1]:>8.4f} {r[2]:>8.1f} {r[3]:>7.1f} {r[4]:>7.1f} {r[5]:>7d}")
print("\nReference: Impulse #1 (deployed): 97.51% test, 1 ms NN + ~900 ms DSP, 1.5 KB RAM, 17.6 KB flash (Studio)")
print("Reference: Impulse #2 (Studio):   98.43% test, 723 ms, 24 KB RAM, 228 KB flash (float32-I/O path)")
print("Pruned model gzipped: 17.7 KB vs 23.3 KB unpruned (sparsity only visible under compression)")

config           int8 acc  file KB  RAM KB  ROM KB  lat ms
full_16_32_64      0.9899     60.0    10.4    89.0      26
half_8_16_32       0.9899     33.7     7.3    63.4      14
quarter_4_8_16     0.9928     22.4     6.3    52.3      15
half_pruned50      0.9884     33.7     7.3    63.4      17
eighth_2_4_8       0.8913     16.8     6.1    47.1      23
minimal_1conv      0.9928     10.6     4.8    45.7       9

Reference: Impulse #1 (deployed): 97.51% test, 1 ms NN + ~900 ms DSP, 1.5 KB RAM, 17.6 KB flash (Studio)
Reference: Impulse #2 (Studio):   98.43% test, 723 ms, 24 KB RAM, 228 KB flash (float32-I/O path)
Pruned model gzipped: 17.7 KB vs 23.3 KB unpruned (sparsity only visible under compression)
